# Developed-Markets External Validation

This notebook evaluates the frozen US research specifications separately in four pre-specified developed markets: the United Kingdom, Australia, Germany, and France. Country selection is based on institutional comparability, complete 2000--2024 coverage, cross-sectional breadth, characteristic availability, and approximately complete next-month-return coverage; no country is selected using model performance.

Each country uses its stable JKP `id` as the security identifier, excludes nano stocks, rank-normalizes characteristics within its own monthly eligible universe, and applies the unchanged 15-year training, 4-year validation, 1-year test schedule. The resulting OOS period is January 2019--December 2024 (72 months).

Three specifications are trained independently in every country: `LGBM_40`, `DEEPSET_40_DYNAMIC`, and `HYBRID_LGBM40_DEEPSET40_DYNAMIC`. A transparent 50/50 comparator is formed by averaging the aligned OOS predictions of the two standalone models; it requires no additional fitting. Every completed country is reported.

## 1. Runtime and project setup

In [ ]:
import gc
import importlib
import os
import sys
from pathlib import Path

LOCAL_PROJECT_DIR = Path(r'C:\Users\sandh\OneDrive\Documents\Coding\FDS Project')
LOCAL_DATA_DIR = Path(r'C:\Users\sandh\FDS Research Project\jkp_developed_153_parquet_2000_2024')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FDS Project')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'jkp_developed_153_parquet_2000_2024'

try:
    from google.colab import drive
except ImportError:
    RUNNING_IN_COLAB = False
    PROJECT_DIR, DATA_DIR = LOCAL_PROJECT_DIR, LOCAL_DATA_DIR
else:
    RUNNING_IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    PROJECT_DIR, DATA_DIR = DRIVE_PROJECT_DIR, DRIVE_DATA_DIR

for required in (PROJECT_DIR, PROJECT_DIR / 'src', DATA_DIR):
    if not required.is_dir():
        raise FileNotFoundError(f'Required directory not found: {required}')
os.chdir(PROJECT_DIR)
project_path = str(PROJECT_DIR)
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)
for module_name in tuple(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import src
if PROJECT_DIR.resolve() not in Path(src.__file__).resolve().parents:
    raise RuntimeError(f'Imported src from the wrong location: {src.__file__}')
print('Project directory:', PROJECT_DIR)
print('Country data directory:', DATA_DIR)

## 2. Frozen country and model configuration

In [ ]:
from src.config import ExperimentConfig, UniverseConfig
from src.developed_markets import COUNTRY_NAMES, EXTERNAL_MODEL_IDS

COUNTRIES = ('GBR', 'AUS', 'DEU', 'FRA')
OUTPUT_DIR = PROJECT_DIR / 'model_runs' / 'developed_markets'
SUMMARY_DIR = OUTPUT_DIR / 'summary'

CONFIGS = {
    country: ExperimentConfig(
        experiment_id=f'external_validation_{country}_v1',
        project_dir=PROJECT_DIR,
        data_path=DATA_DIR / f'jkp_{country}_153_2000_2024.parquet',
        output_dir=OUTPUT_DIR,
        selected_models=EXTERNAL_MODEL_IDS,
        seed=42,
        use_gpu=True,
        universe=UniverseConfig(
            country=country, start_year=2000, end_year=2024,
            security_id_col='id',
        ),
    )
    for country in COUNTRIES
}
for config in CONFIGS.values():
    config.validate()
print({country: str(config.data_path) for country, config in CONFIGS.items()})

## 3. Preflight validation

In [ ]:
import pyarrow.parquet as pq
import torch
from src.config import FEATURES_40
from src.models import MODEL_REGISTRY

required_columns = {
    'id', 'eom', 'excntry', 'size_grp', 'me', 'ret_exc_lead1m', *FEATURES_40
}
for country, config in CONFIGS.items():
    if not config.data_path.is_file():
        raise FileNotFoundError(config.data_path)
    columns = set(pq.read_schema(config.data_path).names)
    missing = sorted(required_columns - columns)
    if missing:
        raise ValueError(f'{country} is missing required columns: {missing}')
    if 'permno' in columns:
        print(f'{country}: PERMNO is present but is not used; JKP id is the security key.')
if set(EXTERNAL_MODEL_IDS) - set(MODEL_REGISTRY):
    raise RuntimeError('An external-validation model is not registered.')
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before model estimation.')
print('GPU:', torch.cuda.get_device_name(0))
print('Country files and model registry: PASS')

## 4. Run or resume country models

Each country and model has an independent experiment directory and signature. Completed annual refits load from disk; incomplete refits resume without altering completed artifacts.

In [ ]:
from dataclasses import replace
from src.runner import ExperimentRunner

for country, country_config in CONFIGS.items():
    print(f'\n######## {country}: {COUNTRY_NAMES[country]} ########')
    for model_id in EXTERNAL_MODEL_IDS:
        print(f'\n=== {country} | {model_id} ===')
        model_config = replace(country_config, selected_models=(model_id,))
        ExperimentRunner(model_config).run()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 5. Transparent 50/50 comparator and country tables

In [ ]:
from IPython.display import display
from src.developed_markets import country_comparison

country_tables = {}
for country, config in CONFIGS.items():
    table = country_comparison(config, include_fifty_fifty=True)
    country_tables[country] = table
    print(f'\n{COUNTRY_NAMES[country]}')
    display(table)

## 6. Portfolio implementability robustness

For every country and specification, the 10% tail portfolio is evaluated under full and ex-microcap universes, equal and value weighting, proportional transaction costs, missing-return stress, and observed-return outlier scenarios.

In [ ]:
import pandas as pd
from src.developed_markets import FIFTY_FIFTY_ID
from src.portfolio_robustness import run_portfolio_robustness

robustness_tables = []
for country, config in CONFIGS.items():
    table = run_portfolio_robustness(
        config.run_dir, model_ids=(*EXTERNAL_MODEL_IDS, FIFTY_FIFTY_ID)
    )
    table.insert(0, 'country_name', COUNTRY_NAMES[country])
    table.insert(0, 'country', country)
    robustness_tables.append(table)
external_robustness = pd.concat(robustness_tables, ignore_index=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
external_robustness.to_csv(
    SUMMARY_DIR / 'developed_markets_portfolio_robustness.csv', index=False
)
display(external_robustness)

## 7. Aggregate cross-country comparison

In [ ]:
from src.developed_markets import aggregate_country_comparisons

external_comparison, hybrid_improvements = aggregate_country_comparisons(
    CONFIGS, SUMMARY_DIR
)
display(external_comparison)
display(hybrid_improvements)

## 8. Completion checks

In [ ]:
expected_months = 72
expected_models_per_country = len(EXTERNAL_MODEL_IDS) + 1
if len(external_comparison) != len(COUNTRIES) * expected_models_per_country:
    raise RuntimeError('The aggregate country/model table is incomplete.')
if external_comparison['n_months'].ne(expected_months).any():
    display(external_comparison.loc[external_comparison['n_months'].ne(expected_months)])
    raise RuntimeError('At least one country/model does not cover all 72 OOS months.')
if len(external_robustness) != len(COUNTRIES) * expected_models_per_country * 4:
    raise RuntimeError('The aggregate implementability table is incomplete.')
print('External validation complete for:', list(COUNTRIES))
print('Model-country rows:', len(external_comparison))
print('OOS period: 2019-01 through 2024-12')
print('Summary directory:', SUMMARY_DIR)